Đọc dữ liệu

In [1]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',', index_col=0)

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 10)

<class 'pandas.DataFrame'>
RangeIndex: 53943 entries, 1 to 53943
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53943 non-null  float64
 1   cut      53943 non-null  str    
 2   color    53943 non-null  str    
 3   clarity  53943 non-null  str    
 4   depth    53943 non-null  float64
 5   table    53943 non-null  float64
 6   price    53943 non-null  int64  
 7   x        53943 non-null  float64
 8   y        53943 non-null  float64
 9   z        53943 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 4.1 MB

--- 5 dòng đầu tiên ---
   carat      cut color clarity  depth  table  price     x     y     z
1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
4   0.29  Premium     I     VS2   

Tiền xử lý

In [2]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
1   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
2   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
3   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
4   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
5   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia dữ liệu

In [3]:
from sklearn.model_selection import KFold

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu thành 5 fold
Kf = KFold(n_splits = 5, shuffle = True, random_state=42)

Training

In [4]:
# Import các thư viện cần thiết
import numpy as np
import pandas as pd
import time
from sklearn.svm import SVR
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

# 1. Định nghĩa không gian tham số cần tìm kiếm 
param_distributions = {
    "svr__kernel": ["rbf"],
    "svr__C": [0.1, 1, 10, 50, 100, 500, 1000, 2000],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10],
    "svr__gamma": ["scale", "auto", 0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1]
}

# 2. Tạo Pipeline tích hợp chuẩn hóa StandardScaler và mô hình SVR
svr_pipeline = make_pipeline(
    StandardScaler(),  # Chuẩn hóa dữ liệu đầu vào
    SVR()              # Khởi tạo mô hình Support Vector Regression
)

# 3. Khởi tạo bộ tìm kiếm tham số ngẫu nhiên kết hợp 5-Fold Cross Validation
random_search = RandomizedSearchCV(
    estimator=svr_pipeline,                   # Pipeline cần tối ưu hóa
    param_distributions=param_distributions,  # Tập tham số cần thử nghiệm
    n_iter=10,                                # Đánh giá ngẫu nhiên 10 bộ tham số khác nhau để tiết kiệm thời gian
    cv=Kf,                                    # Sử dụng bộ chia 5-Fold (Kf) đã định nghĩa từ trước
    scoring="neg_root_mean_squared_error",    # Tối ưu hóa dựa trên RMSE
    random_state=42,                          # Thiết lập hạt giống ngẫu nhiên để cố định kết quả
    n_jobs=-1,                                # Sử dụng toàn bộ số nhân CPU khả dụng để tăng tốc
    refit=False                               # Không tự động huấn luyện lại toàn bộ dữ liệu sau khi tìm xong
)

# 4. Ghi nhận thời gian bắt đầu tìm kiếm
search_start = time.time()

# 5. Tiến hành quét và tìm kiếm tham số tối ưu trên dữ liệu
random_search.fit(X, y)

# 6. Tính tổng thời gian thực hiện tìm kiếm siêu tham số
total_search_time = time.time() - search_start

# 7. Trích xuất bộ tham số tối ưu nhất được tìm thấy
best_params = random_search.best_params_

# 8. Sao chép Pipeline ban đầu và gán bộ tham số tốt nhất vào mô hình mới
best_svr_model = clone(svr_pipeline).set_params(**best_params)

# 9. Khởi tạo danh sách chứa kết quả đo lường của từng Fold
results = []
total_training_time = 0

# 10. Tiến hành huấn luyện và đánh giá mô hình tối ưu qua 5-Fold Cross Validation
for fold, (train_index, test_index) in enumerate(Kf.split(X), 1):
    # Phân tách tập dữ liệu huấn luyện cho Fold hiện tại
    X_train_fold = X.iloc[train_index]
    y_train_fold = y.iloc[train_index]
    
    # Phân tách tập dữ liệu kiểm thử cho Fold hiện tại
    X_test_fold = X.iloc[test_index]
    y_test_fold = y.iloc[test_index]
    
    # Ghi nhận mốc thời gian bắt đầu huấn luyện Fold
    fold_start_time = time.time()
    
    # Huấn luyện mô hình tối ưu trên tập dữ liệu của Fold hiện tại
    best_svr_model.fit(X_train_fold, y_train_fold)
    
    # Tính toán thời gian huấn luyện của Fold này
    fold_train_time = time.time() - fold_start_time
    total_training_time += fold_train_time
    
    # Thực hiện dự đoán giá trị trên tập Train và tập Test của Fold
    y_pred_train = best_svr_model.predict(X_train_fold)
    y_pred_test = best_svr_model.predict(X_test_fold)
    
    # Tính toán các chỉ số đánh giá sai số trên tập Train
    mae_train = mean_absolute_error(y_train_fold, y_pred_train)
    mse_train = mean_squared_error(y_train_fold, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train_fold, y_pred_train)
    
    # Tính toán các chỉ số đánh giá sai số trên tập Test
    mae_test = mean_absolute_error(y_test_fold, y_pred_test)
    mse_test = mean_squared_error(y_test_fold, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test_fold, y_pred_test)
    
    # Lưu trữ toàn bộ kết quả của Fold hiện tại vào danh sách chung
    results.append([
        fold,
        mae_train, mse_train, rmse_train, r2_train,
        mae_test, mse_test, rmse_test, r2_test,
        fold_train_time
    ])
    
    # In chi tiết kết quả từng Fold ra màn hình
    print(f"\n========== FOLD {fold} ==========")
    print(f"Số mẫu Train: {len(X_train_fold)}")
    print(f"Số mẫu Test: {len(X_test_fold)}")
    print("\n--- Kết quả Train ---")
    print(f"MAE Train:  {mae_train:.4f}")
    print(f"MSE Train:  {mse_train:.4f}")
    print(f"RMSE Train: {rmse_train:.4f}")
    print(f"R2 Train:   {r2_train:.4f}")
    print("\n--- Kết quả Test ---")
    print(f"MAE Test:   {mae_test:.4f}")
    print(f"MSE Test:   {mse_test:.4f}")
    print(f"RMSE Test:  {rmse_test:.4f}")
    print(f"R2 Test:    {r2_test:.4f}")
    print(f"\nThời gian Train: {fold_train_time:.4f} giây")

# 11. Đóng gói kết quả của các Fold vào DataFrame để xử lý thống kê
columns = [
    "Fold",
    "MAE Train", "MSE Train", "RMSE Train", "R2 Train",
    "MAE Test", "MSE Test", "RMSE Test", "R2 Test",
    "Training Time"
]
results_df = pd.DataFrame(results, columns=columns)

# 12. Tính toán giá trị trung bình các chỉ số qua 5 Fold
mean_results = results_df.drop(columns=["Fold"]).mean()

# 13. Hiển thị thông tin kết quả tối ưu hóa tham số SVR
print("\n=================== THAM SỐ SVR TỐT NHẤT ===================")
print(f"Kernel được chọn: {best_params['svr__kernel']}")
print(f"Tham số C tối ưu: {best_params['svr__C']}")
print(f"Tham số Epsilon tối ưu: {best_params['svr__epsilon']}")
print(f"Tham số Gamma tối ưu: {best_params['svr__gamma']}")
print(f"RMSE tốt nhất thu được khi Tuning: {-random_search.best_score_:.4f}")
print(f"Tổng thời gian tìm kiếm tham số: {total_search_time:.4f} giây")

# 14. Hiển thị báo cáo kết quả đánh giá trung bình 5 Fold
print("\n========== KẾT QUẢ TRUNG BÌNH 5 FOLD ==========")
print("\n--- Trung bình Train ---")
print(f"MAE Train:  {mean_results['MAE Train']:.4f}")
print(f"MSE Train:  {mean_results['MSE Train']:.4f}")
print(f"RMSE Train: {mean_results['RMSE Train']:.4f}")
print(f"R2 Train:   {mean_results['R2 Train']:.4f}")
print("\n--- Trung bình Test ---")
print(f"MAE Test:   {mean_results['MAE Test']:.4f}")
print(f"MSE Test:   {mean_results['MSE Test']:.4f}")
print(f"RMSE Test:  {mean_results['RMSE Test']:.4f}")
print(f"R2 Test:    {mean_results['R2 Test']:.4f}")
print(f"\nThời gian Train trung bình: {mean_results['Training Time']:.4f} giây")
print(f"Tổng thời gian Train qua 5 Fold: {total_training_time:.4f} giây")


========== FOLD 1 ==========
Số mẫu Train: 43020
Số mẫu Test: 10755

--- Kết quả Train ---
MAE Train:  350.3721
MSE Train:  529049.3745
RMSE Train: 727.3578
R2 Train:   0.9667

--- Kết quả Test ---
MAE Test:   359.2089
MSE Test:   559651.1071
RMSE Test:  748.0983
R2 Test:    0.9646

Thời gian Train: 435.8410 giây

========== FOLD 2 ==========
Số mẫu Train: 43020
Số mẫu Test: 10755

--- Kết quả Train ---
MAE Train:  350.1883
MSE Train:  529273.0649
RMSE Train: 727.5116
R2 Train:   0.9669

--- Kết quả Test ---
MAE Test:   355.4590
MSE Test:   538765.5908
RMSE Test:  734.0065
R2 Test:    0.9654

Thời gian Train: 112.4487 giây

========== FOLD 3 ==========
Số mẫu Train: 43020
Số mẫu Test: 10755

--- Kết quả Train ---
MAE Train:  353.3823
MSE Train:  538875.6638
RMSE Train: 734.0815
R2 Train:   0.9661

--- Kết quả Test ---
MAE Test:   344.2640
MSE Test:   495785.9144
RMSE Test:  704.1207
R2 Test:    0.9687

Thời gian Train: 77.2602 giây

========== FOLD 4 ==========
Số mẫu Train: 43020
Số 

KeyboardInterrupt: 